<a href="https://colab.research.google.com/github/Orth33/loan-approval-prediction/blob/main/Loan_Approval_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Initial Setup and Data Loading
We will import the necessary libraries to manipulate the data and eventually build our model. We will also load our dataset to inspect its structure.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("architsharma01/loan-approval-prediction-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'loan-approval-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/loan-approval-prediction-dataset


In [3]:
# Load the dataset (Ensure the CSV is uploaded to your Colab session)
df = pd.read_csv('/kaggle/input/loan-approval-prediction-dataset/loan_approval_dataset.csv')

# Strip leading/trailing spaces from column names (common in Kaggle datasets)
df.columns = df.columns.str.strip()

# Display the first 5 rows to understand what we are working with
display(df.head())

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


# 2. Data Cleaning and Preprocessing

- ID Removal: We will drop the loan_id column because an arbitrary identification number has no predictive value.

- Missing Values: We will fill numerical missing values with the median, and categorical missing values with the mode (most frequent value).

- Encoding: We will convert our target variable (loan_status) to 1s and 0s, and use One-Hot Encoding for our categorical features.

In [5]:
print(df.describe())

           loan_id  no_of_dependents  income_annum   loan_amount    loan_term  \
count  4269.000000       4269.000000  4.269000e+03  4.269000e+03  4269.000000   
mean   2135.000000          2.498712  5.059124e+06  1.513345e+07    10.900445   
std    1232.498479          1.695910  2.806840e+06  9.043363e+06     5.709187   
min       1.000000          0.000000  2.000000e+05  3.000000e+05     2.000000   
25%    1068.000000          1.000000  2.700000e+06  7.700000e+06     6.000000   
50%    2135.000000          3.000000  5.100000e+06  1.450000e+07    10.000000   
75%    3202.000000          4.000000  7.500000e+06  2.150000e+07    16.000000   
max    4269.000000          5.000000  9.900000e+06  3.950000e+07    20.000000   

       cibil_score  residential_assets_value  commercial_assets_value  \
count  4269.000000              4.269000e+03             4.269000e+03   
mean    599.936051              7.472617e+06             4.973155e+06   
std     172.430401              6.503637e+06       

In [6]:
# Drop ID column if it exists
if 'loan_id' in df.columns:
    df = df.drop('loan_id', axis=1)

# Handle Missing Values
for col in df.columns:
    if df[col].dtype == 'object':
        # Fill missing text categories with the most frequent value
        df[col].fillna(df[col].mode()[0], inplace=True)
        # Strip hidden whitespace from strings
        df[col] = df[col].str.strip()
    else:
        # Fill missing numbers with the median
        df[col].fillna(df[col].median(), inplace=True)

# Define target variable (assuming it's named 'loan_status')
target_col = 'loan_status'

# Separate features (X) and target (y)
X = df.drop(target_col, axis=1)
y = df[target_col]

# Encode Target Variable (e.g., 'Approved' -> 1, 'Rejected' -> 0)
le = LabelEncoder()
y = le.fit_transform(y)

# Encode Categorical Features (One-Hot Encoding)
X = pd.get_dummies(X, drop_first=True)

print(f"Data shape after encoding: {X.shape}")

Data shape after encoding: (4269, 11)


/tmp/ipython-input-1828613703.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipython-input-1828613703.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usi

# 3. Train-Test Split and Model Training
We will split our data so that 80% is used to train the model, and 20% is held back to test it. We will train a Decision Tree model on the natural (imbalanced) dataset to see how it performs without any advanced balancing techniques.

In [7]:
# Split the data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check the imbalance in the training data
print("Class distribution in training data:")
print(pd.Series(y_train).value_counts(normalize=True))

# Scale the features (Important for models like Logistic Regression, good practice overall)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the classification model
model = DecisionTreeClassifier(random_state=42, max_depth=5)
model.fit(X_train_scaled, y_train)

# Generate predictions on the test set
y_pred = model.predict(X_test_scaled)

Class distribution in training data:
0    0.620791
1    0.379209
Name: proportion, dtype: float64


# 4. Model Evaluation
We will use a classification report to evaluate the model's true performance.

- Precision: Out of all the loans the model predicted as Approved, how many were actually Approved?

- Recall: Out of all the loans that were actually Approved, how many did the model find?

- F1-Score: The harmonic mean of precision and recall (the best overall indicator of success on imbalanced data).

In [8]:
# Print the full classification report
print("Classification Report (Imbalanced Data):\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Print specific metrics for clarity
print("--- Specific Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred):.2f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.2f}")

Classification Report (Imbalanced Data):

              precision    recall  f1-score   support

    Approved       0.98      0.96      0.97       536
    Rejected       0.94      0.97      0.96       318

    accuracy                           0.97       854
   macro avg       0.96      0.97      0.97       854
weighted avg       0.97      0.97      0.97       854

--- Specific Metrics ---
Precision: 0.94
Recall:    0.97
F1-Score:  0.96


# 5. Bonus: Applying SMOTE and Comparing Models
Although our baseline model performed well, real-world financial data is often heavily skewed, and a model might struggle to identify the minority class.

To address this, we will use SMOTE (Synthetic Minority Over-sampling Technique). SMOTE generates synthetic data points for the minority class so that our model trains on a perfectly balanced 50/50 dataset.

- Crucial Step: We only apply SMOTE to the training data. If we apply it to the test data, we would be evaluating the model on "fake" data, which invalidates our metrics.

After balancing the data, we will train and compare two different algorithms: a Decision Tree and Logistic Regression.

In [10]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

# 1. Apply SMOTE to the training data ONLY
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Display the class distribution before and after SMOTE
print("Class distribution BEFORE SMOTE:")
print(pd.Series(y_train).value_counts())

print("\nClass distribution AFTER SMOTE:")
print(pd.Series(y_train_smote).value_counts())

Class distribution BEFORE SMOTE:
0    2120
1    1295
Name: count, dtype: int64

Class distribution AFTER SMOTE:
0    2120
1    2120
Name: count, dtype: int64


In [11]:
# 2. Train a Decision Tree on the SMOTE-balanced data
dt_smote = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_smote.fit(X_train_smote, y_train_smote)
y_pred_dt_smote = dt_smote.predict(X_test_scaled)

# 3. Train a Logistic Regression model on the SMOTE-balanced data
log_reg_smote = LogisticRegression(random_state=42)
log_reg_smote.fit(X_train_smote, y_train_smote)
y_pred_log_smote = log_reg_smote.predict(X_test_scaled)

# 4. Evaluate and Compare the Models
print("\n" + "="*40)
print("=== Decision Tree (Trained with SMOTE) ===")
print("="*40)
print(classification_report(y_test, y_pred_dt_smote, target_names=le.classes_))

print("\n" + "="*45)
print("=== Logistic Regression (Trained with SMOTE) ===")
print("="*45)
print(classification_report(y_test, y_pred_log_smote, target_names=le.classes_))


=== Decision Tree (Trained with SMOTE) ===
              precision    recall  f1-score   support

    Approved       0.98      0.97      0.98       536
    Rejected       0.95      0.97      0.96       318

    accuracy                           0.97       854
   macro avg       0.96      0.97      0.97       854
weighted avg       0.97      0.97      0.97       854


=== Logistic Regression (Trained with SMOTE) ===
              precision    recall  f1-score   support

    Approved       0.95      0.91      0.93       536
    Rejected       0.85      0.91      0.88       318

    accuracy                           0.91       854
   macro avg       0.90      0.91      0.90       854
weighted avg       0.91      0.91      0.91       854



In [13]:
# Print specific metrics for clarity
print("--- Decision Tree Specific Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred_dt_smote):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt_smote):.2f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt_smote):.2f}")

# Print specific metrics for clarity
print("\n--- Logistic Regression Specific Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred_log_smote):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred_log_smote):.2f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_log_smote):.2f}")

--- Decision Tree Specific Metrics ---
Precision: 0.95
Recall:    0.97
F1-Score:  0.96

--- Logistic Regression Specific Metrics ---
Precision: 0.85
Recall:    0.91
F1-Score:  0.88
